# 1. Import Libraries


In [1]:
import pandas as pd
import numpy as np
import sqlite3
import json

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from scipy import stats

# 2. Load Data

In [7]:
import pandas as pd
import sqlite3
import json

# -----------------------------
# USERS
# -----------------------------
users = pd.read_excel("users.xlsx")
print(users.head())

  user_id           name  age gender       city registration_date
0   U0001  Vihaan Sharma   35  Other     Jaipur        2022-09-08
1   U0002      Sai Reddy   30  Other  Hyderabad        2023-11-24
2   U0003   Aarohi Gupta   37  Other     Indore        2022-02-02
3   U0004    Aarav Gupta   44   Male    Kolkata        2023-06-02
4   U0005    Sara Sharma   30  Other    Chennai        2024-01-04


In [8]:
# -----------------------------
# SALES
# -----------------------------
with open("sales.json") as f:
    sales = pd.DataFrame(json.load(f))
    print(sales.head())

  transaction_id user_id product_id  amount payment_type        date
0        T000001   U0024       P015   67.67       Wallet  2023-02-12
1        T000002   U0196       P044   76.44          UPI  2023-03-24
2        T000003   U0196       P049  104.57   Debit Card  2025-08-21
3        T000004   U0133       P042  102.75  Net Banking  2024-07-23
4        T000005   U0047       P038   23.89  Net Banking  2025-10-04


In [3]:
pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.


In [10]:
import mysql.connector
import pandas as pd

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="1610",
    database="product_db"
)

query = "SELECT * FROM product"
product = pd.read_sql(query, conn)

print(product.head())

  product_id product_name category    price  stock
0       P001  Product_001  Grocery   264.89    371
1       P002  Product_002  Grocery   605.91    150
2       P003  Product_003   Beauty  3027.98    127
3       P004  Product_004     Toys  2600.12    229
4       P005  Product_005    Books  1178.99     18


C:\Users\macwan alex\AppData\Local\Temp\ipykernel_16944\2275830670.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  product = pd.read_sql(query, conn)


# 3.DATA UNDERSTANDING

In [11]:
print(users.info())
print(sales.info())
print(product.info())

print(sales.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   user_id            200 non-null    object        
 1   name               200 non-null    object        
 2   age                200 non-null    int64         
 3   gender             200 non-null    object        
 4   city               200 non-null    object        
 5   registration_date  200 non-null    datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(4)
memory usage: 9.5+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   transaction_id  1000 non-null   object 
 1   user_id         1000 non-null   object 
 2   product_id      1000 non-null   object 
 3   amount          1000 non-null   float64
 4   pa

# 4. DATA CLEANING

In [14]:
# Fix date
sales['date'] = pd.to_datetime(sales['date'], errors='coerce')

# Remove invalid values
sales = sales[sales['amount'] > 0]

# Missing values
imputer = SimpleImputer(strategy='mean')
sales[['amount']] = imputer.fit_transform(sales[['amount']])
sales[['amount']]

,amount
0,67.67
1,76.44
2,104.57
3,102.75
4,23.89
...,...
995,71.11
996,53.96
997,76.06
998,62.45


# 5. OUTLIER HANDLING

In [16]:
# IQR method
Q1 = sales['amount'].quantile(0.25)
Q3 = sales['amount'].quantile(0.75)
IQR = Q3 - Q1

sales = sales[
    (sales['amount'] >= Q1 - 1.5*IQR) &
    (sales['amount'] <= Q3 + 1.5*IQR)
]
sales

,transaction_id,user_id,product_id,amount,payment_type,date
0,T000001,U0024,P015,67.67,Wallet,2023-02-12
1,T000002,U0196,P044,76.44,UPI,2023-03-24
2,T000003,U0196,P049,104.57,Debit Card,2025-08-21
3,T000004,U0133,P042,102.75,Net Banking,2024-07-23
4,T000005,U0047,P038,23.89,Net Banking,2025-10-04
...,...,...,...,...,...,...
995,T000996,U0178,P043,71.11,Credit Card,2025-02-20
996,T000997,U0100,P027,53.96,Wallet,2024-10-02
997,T000998,U0142,P004,76.06,Credit Card,2024-05-29
998,T000999,U0052,P040,62.45,Net Banking,2024-04-06


# 6. DATA TRANSFORMATION

In [18]:
# Date features
sales['day'] = sales['date'].dt.day
sales['month'] = sales['date'].dt.month
sales['year'] = sales['date'].dt.year

# Encoding
le = LabelEncoder()
sales['payment_type'] = le.fit_transform(sales['payment_type'])

# Binning
sales['spending_category'] = pd.cut(
    sales['amount'],
    bins=[0,50,150,1000],
    labels=['Low','Medium','High']
)

# Log transform
sales['log_amount'] = np.log1p(sales['amount'])
sales

,transaction_id,user_id,product_id,amount,payment_type,date,day,month,year,spending_category,log_amount
0,T000001,U0024,P015,67.67,5,2023-02-12,12,2,2023,Medium,4.229312
1,T000002,U0196,P044,76.44,4,2023-03-24,24,3,2023,Medium,4.349503
2,T000003,U0196,P049,104.57,2,2025-08-21,21,8,2025,Medium,4.659374
3,T000004,U0133,P042,102.75,3,2024-07-23,23,7,2024,Medium,4.641984
4,T000005,U0047,P038,23.89,3,2025-10-04,4,10,2025,Low,3.214466
...,...,...,...,...,...,...,...,...,...,...,...
995,T000996,U0178,P043,71.11,1,2025-02-20,20,2,2025,Medium,4.278193
996,T000997,U0100,P027,53.96,5,2024-10-02,2,10,2024,Medium,4.006606
997,T000998,U0142,P004,76.06,1,2024-05-29,29,5,2024,Medium,4.344584
998,T000999,U0052,P040,62.45,3,2024-04-06,6,4,2024,Medium,4.150252


# 7. FEATURE SCALING

In [23]:
scaler = StandardScaler()
sales['amount_scaled'] = scaler.fit_transform(sales[['amount']])
minmax = MinMaxScaler()
sales['amount_minmax'] = minmax.fit_transform(sales[['amount']])
sales

,transaction_id,user_id,product_id,amount,payment_type,date,day,month,year,spending_category,log_amount,amount_scaled,amount_minmax
0,T000001,U0024,P015,67.67,5,2023-02-12,12,2,2023,Medium,4.229312,0.295705,0.448087
1,T000002,U0196,P044,76.44,4,2023-03-24,24,3,2023,Medium,4.349503,0.596206,0.513736
2,T000003,U0196,P049,104.57,2,2025-08-21,21,8,2025,Medium,4.659374,1.560073,0.724306
3,T000004,U0133,P042,102.75,3,2024-07-23,23,7,2024,Medium,4.641984,1.497711,0.710682
4,T000005,U0047,P038,23.89,3,2025-10-04,4,10,2025,Low,3.214466,-1.204404,0.120368
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,T000996,U0178,P043,71.11,1,2025-02-20,20,2,2025,Medium,4.278193,0.413575,0.473838
996,T000997,U0100,P027,53.96,5,2024-10-02,2,10,2024,Medium,4.006606,-0.174064,0.345460
997,T000998,U0142,P004,76.06,1,2024-05-29,29,5,2024,Medium,4.344584,0.583186,0.510892
998,T000999,U0052,P040,62.45,3,2024-04-06,6,4,2024,Medium,4.150252,0.116843,0.409013


# 8. FEATURE ENGINEERING

In [25]:
# Average spend
avg_spend = sales.groupby('user_id')['amount'].mean().reset_index()
avg_spend.rename(columns={'amount':'avg_spend'}, inplace=True)

# Frequency
freq = sales.groupby('user_id').size().reset_index(name='purchase_freq')

# Recency
last = sales.groupby('user_id')['date'].max().reset_index()
last['days_since_last'] = (pd.Timestamp.today() - last['date']).dt.days

# Category spend
cat_spend = sales.groupby(['user_id','spending_category'])['amount'].sum().unstack()
cat_spend

C:\Users\macwan alex\AppData\Local\Temp\ipykernel_16944\865591624.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cat_spend = sales.groupby(['user_id','spending_category'])['amount'].sum().unstack()


spending_category,Low,Medium,High
user_id,,,
U0001,87.36,0.00,0.0
U0002,77.69,311.35,0.0
U0003,103.52,58.87,0.0
U0004,0.00,135.07,0.0
U0005,154.52,193.17,0.0
...,...,...,...
U0196,49.89,407.75,0.0
U0197,82.16,162.26,0.0
U0198,31.38,129.77,0.0


# 9. FINAL MERGE

In [36]:
cat_spend = cat_spend.reset_index()

In [37]:
# Category spend
cat_spend = sales.groupby(['user_id','spending_category'])['amount'].sum().unstack()

# FIX (IMPORTANT)
cat_spend = cat_spend.reset_index()

C:\Users\macwan alex\AppData\Local\Temp\ipykernel_16944\865227394.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cat_spend = sales.groupby(['user_id','spending_category'])['amount'].sum().unstack()


In [38]:
final = users.merge(avg_spend, on='user_id', how='left')
final = final.merge(freq, on='user_id', how='left')
final = final.merge(last[['user_id','days_since_last']], on='user_id', how='left')
final = final.merge(cat_spend, on='user_id', how='left')
final

,user_id,name,age,gender,city,registration_date,avg_spend,purchase_freq,days_since_last,Low,Medium,High
0,U0001,Vihaan Sharma,35,Other,Jaipur,2022-09-08,43.680000,2,618,87.36,0.00,0.0
1,U0002,Sai Reddy,30,Other,Hyderabad,2023-11-24,64.840000,6,733,77.69,311.35,0.0
2,U0003,Aarohi Gupta,37,Other,Indore,2022-02-02,40.597500,4,180,103.52,58.87,0.0
3,U0004,Aarav Gupta,44,Male,Kolkata,2023-06-02,67.535000,2,757,0.00,135.07,0.0
4,U0005,Sara Sharma,30,Other,Chennai,2024-01-04,57.948333,6,312,154.52,193.17,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
195,U0196,Kabir Kulkarni,35,Other,Patna,2024-08-01,76.273333,6,252,49.89,407.75,0.0
196,U0197,Kabir Roy,24,Male,Thane,2023-12-07,61.105000,4,186,82.16,162.26,0.0
197,U0198,Kabir Bose,33,Female,Visakhapatnam,2023-07-07,53.716667,3,183,31.38,129.77,0.0
198,U0199,Meera Roy,32,Male,Ghaziabad,2022-02-10,53.633750,8,277,139.45,289.62,0.0


# 10. REPORT

In [33]:
print("Records:", final.shape[0])
print("Features:", final.shape[1])
print("Missing values:\n", final.isnull().sum())

Records: 200
Features: 9
Missing values:
 user_id              0
name                 0
age                  0
gender               0
city                 0
registration_date    0
avg_spend            0
purchase_freq        0
days_since_last      0
dtype: int64


# 11. SAVE FILE

In [34]:
final.to_csv("final_cleaned_dataset.csv", index=False)